# Aralin 13 - Memorya ng Ahente


## Setup

Ipinapakita ng notebook na ito kung paano bumuo ng ahente sa pag-book ng biyahe gamit ang **persistent memory** gamit ang **Microsoft Agent Framework** (MAF).

Malalaman mo kung paano naaapektuhan ng iba't ibang uri ng memorya ng ahente — working, short-term, at long-term — ang paraan ng pag-alala at paggamit ng impormasyon ng ahente sa mga pag-uusap.

**Mga Kinakailangan:**
- Isang Microsoft Foundry project na may deployed chat model (hal. `gpt-5-mini`).
- Nakalog-in gamit ang Azure CLI — patakbuhin ang `az login` sa iyong terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — ang iyong Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Mga Uri ng Memorya ng Ahente

Maaaring gumamit ang mga AI agent ng iba't ibang uri ng memorya, na bawat isa ay may kanya-kanyang layunin:

### Working Memory
Ang mismong thread ng pag-uusap — ang mga mensahe na ipinagpalitan sa isang session. Maaaring balikan ng ahente ang mga naunang mensahe sa parehong thread upang mapanatili ang pagkakaugnay. Sa MAF ito ay nililikha sa pamamagitan ng **`agent.create_session()`**, na nagbabalik ng `AgentSession`.

### Short-Term Memory
Impormasyon na nananatili habang tumatagal ang isang gawain o session ngunit hindi permanente. Halimbawa, maaaring mangolekta ang ahente ng mga katotohanan sa isang multi-turn planning na pag-uusap at gamitin ang mga ito upang makagawa ng panghuling itinerary.

### Long-Term Memory
Mga kagustuhan at katotohanan na nananatili **sa paglipas ng mga session**. Hindi na kailangang ulitin ng isang bumabalik na user ang kanilang mga dietary restrictions o estilo ng paglalakbay. Karaniwang sinusuportahan ang long-term memory ng isang panlabas na imbakan — isang database, file, o vector index — at ipinapakita sa ahente sa pamamagitan ng mga tool.


## Gumaganang Memorya gamit ang Mga Session

Ang pinakasimpleng anyo ng memorya ay ang sesyon ng pag-uusap. Kapag ipinasa mo ang parehong sesyon (na nilikha gamit ang `agent.create_session()`) sa mga sunud-sunod na tawag ng `agent.run()`, nakikita ng ahente ang buong kasaysayan ng pag-uusap na iyon at maaaring maalala ang mga naunang detalye.

Gumawa tayo ng ahente ng paglalakbay at ipapakita ang gumaganang memorya.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Tama na naalala ng ahente ang badyet dahil parehong session ang pinagdaanan ng dalawang mensahe. Ito ay **gumaganang memorya** — umiiral lamang ito sa habang buhay ng session.

### Ano ang nangyayari sa isang bagong thread?

Kung gagawa tayo ng **bagong** session, wala nang alaala ng ahente tungkol sa nakaraang usapan:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pangmatagalang Memorya na Pattern

Upang maalala ang mga kagustuhan ng gumagamit **sa pagitan ng mga sesyon**, kailangan natin ng isang tuloy-tuloy na imbakan na nananatili sa labas ng usapan. Ang ahente ay nakakakuha ng access sa imbakan na ito sa pamamagitan ng **mga kasangkapan** — mga function na maaari nitong tawagin upang mag-imbak at kumuha ng impormasyon.

Sa ibaba ay ipinatutupad namin ang isang simpleng in-memory preference store (sa produksyon gagamit ka ng database o vector index bilang back-end) at ilalantad ito bilang mga kasangkapan na maaaring gamitin ng ahente.

### Arkitektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Unang beses na gumagamit na nagbu-book ng biyahe para sa anibersaryo

Si Sarah ay bumisita sa unang pagkakataon. Dapat i-save ng ahente ang kanyang mga kagustuhan gamit ang mga kasangkapan at gamitin ito upang magrekomenda ng mga hotel.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Senaryo 2 — Bumalik si Sarah makalipas ang ilang linggo

Nagsisimula si Sarah ng **bagong thread** (pinapalagay na bagong session). Walang laman ang working memory, ngunit naroroon pa rin sa long-term preference store ang kanyang impormasyon. Dapat itong kunin ng ahente at gamitin upang ipersonalisa ang mga rekomendasyon.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Buod

Sa araling ito, natutunan mo ang tatlong uri ng memorya ng ahente at kung paano ito ipatupad gamit ang Microsoft Agent Framework:

| Uri ng Memorya | Mekanismo ng MAF | Haba ng Panahon |
|---|---|---|
| **Working** | `agent.create_session()` | Isang pag-uusap |
| **Short-term** | Naipon na konteksto sa loob ng isang thread | Isang gawain / sesyon |
| **Long-term** | Panlabas na imbakan na ina-access gamit ang mga function na `@tool` | Sa iba't ibang mga sesyon |

### Mga Pangunahing Punto
1. **`agent.create_session()`** ay nagbibigay ng working memory — nakikita ng ahente ang buong kasaysayan ng pag-uusap sa loob ng isang sesyon.
2. **Nawawala ang konteksto sa mga bagong sesyon** — kung walang long-term memory, hindi maalala ng ahente ang mga nakaraang pag-uusap.
3. **Ang mga function na `@tool` ay nag-uugnay** — pinapayagan nila ang ahente na mag-imbak at kumuha ng impormasyon mula sa matagalang imbakan.
4. **Lumalala ang personalisasyon sa paglipas ng panahon** — habang mas marami ang mga naka-imbak na kagustuhan, mas maganda ang mga rekomendasyon ng ahente.

### Mga Praktikal na Aplikasyon
- **Serbisyo sa Customer**: Tandaan ang kasaysayan at mga kagustuhan ng customer
- **Personal na Katulong**: Panatilihin ang konteksto sa loob ng mga araw o linggo
- **Pangangalaga sa Kalusugan**: Subaybayan ang impormasyon at kagustuhan ng pasyente
- **E-commerce**: Personal na pamimili base sa kasaysayan

### Mga Susunod na Hakbang
- Palitan ang in-memory dict ng database o vector store (hal. Azure AI Search)
- Magdagdag ng memory expiration para sa mga sensitibong impormasyon sa oras
- Bumuo ng mga multi-agent system na may pinagbabahaging memorya
- Tuklasin ang Cognee notebook para sa knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
